In [13]:
import pandas as pd 
import numpy as np
import plotly.io as pio
pio.renderers.default = "browser"
import matplotlib.pyplot as plt
import seaborn as sns 
from datetime import datetime
import plotly.graph_objects as go 
from plotly.subplots import make_subplots
import warnings


warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
sns.set(palette='husl')

print('Everthing is Fine')

Everthing is Fine


In [14]:
df = pd.read_csv('funnel_analysis_data.csv')

print('---Top 5 rows ---')
print(df.head())

print('\n --- last 5 rows ---')
print(df.tail())

print('\n-- Random Sample ---')
print(df.sample(5))

print('\n--- Column ---')
print(df.columns)
print("Total Number Of Column:", len(df.columns))

print("\n--- Data Types ---")
print(df.dtypes)

print("\n--- Info --")
df.info()

print("\n--- Describe---")
print(df.describe())

---Top 5 rows ---
    User_ID    Session_ID            Timestamp        Event   Device Region  \
0  USR00001  SESS3e5852bc  2026-03-12 04:49:51       Browse  Desktop  North   
1  USR00001  SESS3e5852bc  2026-03-12 04:52:51  Add to Cart  Desktop  North   
2  USR00001  SESS3e5852bc  2026-03-12 04:57:51     Checkout  Desktop  North   
3  USR00001  SESS3e5852bc  2026-03-12 04:59:51     Purchase  Desktop  North   
4  USR00002  SESS6b780e87  2026-04-07 14:53:34       Browse   Laptop  South   

      Channel Product_category  Revenue  
0  Google Ads           Sports     0.00  
1  Google Ads           Sports     0.00  
2  Google Ads           Sports     0.00  
3  Google Ads           Sports  1312.57  
4  Google Ads           Sports     0.00  

 --- last 5 rows ---
        User_ID    Session_ID            Timestamp        Event  Device  \
25014  USR09999  SESS6d48e000  2026-04-02 17:01:35  Add to Cart  Laptop   
25015  USR09999  SESS6d48e000  2026-04-02 17:06:35     Checkout  Laptop   
25016  U

In [15]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

print("Finding Null values:\n",df.isnull().sum())
print("\nFinding Duplicates Values:\n",df.duplicated().sum())
print("\n Total Unique data:\n", df.nunique())


df['Event_Sequence'] = df.groupby('Session_ID').cumcount()+1

print("\n",df.head())

Finding Null values:
 User_ID             0
Session_ID          0
Timestamp           0
Event               0
Device              0
Region              0
Channel             0
Product_category    0
Revenue             0
dtype: int64

Finding Duplicates Values:
 0

 Total Unique data:
 User_ID             10000
Session_ID          10000
Timestamp           24919
Event                   4
Device                  3
Region                  4
Channel                 4
Product_category        5
Revenue              2973
dtype: int64

     User_ID    Session_ID           Timestamp        Event   Device Region  \
0  USR00001  SESS3e5852bc 2026-03-12 04:49:51       Browse  Desktop  North   
1  USR00001  SESS3e5852bc 2026-03-12 04:52:51  Add to Cart  Desktop  North   
2  USR00001  SESS3e5852bc 2026-03-12 04:57:51     Checkout  Desktop  North   
3  USR00001  SESS3e5852bc 2026-03-12 04:59:51     Purchase  Desktop  North   
4  USR00002  SESS6b780e87 2026-04-07 14:53:34       Browse   Laptop  South 

In [17]:
funnel_stages = ['Browse','Add to Cart','Checkout','Purchase']

session_summary = df.groupby('Session_ID').agg(
    User_ID = ('User_ID','first'),
    Start_time = ('Timestamp','min'),
    End_time=('Timestamp','max'),
    Events = ('Event', lambda x: list(x)),
    Device = ('Device','first'),
    Region = ('Region','first'),
    Channel = ('Channel','first'),
    Product_Category = ('Product_category','first'),
    Revenue = ('Revenue','max'),
).reset_index()

session_summary['Session_Duration'] = (session_summary['End_time'] - session_summary['Start_time']).dt.total_seconds() / 60 

def get_max_funnel_stage(events):
    stage_values = {stage: i for i, stage in enumerate(funnel_stages)}
    max_stage_index = -1
    for event in events:
        if event in stage_values and stage_values[event] > max_stage_index:
            max_stage_index = stage_values[event]
    return funnel_stages[max_stage_index] if max_stage_index != -1 else 'Browse'
    
session_summary['Max_funnel_Stage'] = session_summary['Events'].apply(get_max_funnel_stage)
session_summary['Boounce_Flag'] = session_summary['Max_funnel_Stage'].apply(lambda x: 'Yes' if x == 'Browse' else 'No')

print("Session Summary Created")
print(session_summary.head())

Session Summary Created
     Session_ID   User_ID          Start_time            End_time  \
0  SESS000275f2  USR08308 2026-04-01 00:08:52 2026-04-01 00:18:52   
1  SESS0005d0d3  USR03254 2026-03-11 19:51:10 2026-03-11 20:01:10   
2  SESS000caed5  USR00773 2026-03-25 19:34:24 2026-03-25 19:38:24   
3  SESS000f37cd  USR02196 2026-04-08 23:48:18 2026-04-08 23:59:18   
4  SESS001340dd  USR06633 2026-03-29 05:55:19 2026-03-29 05:55:19   

                                      Events   Device Region       Channel  \
0  [Browse, Add to Cart, Checkout, Purchase]   Laptop  South         Email   
1  [Browse, Add to Cart, Checkout, Purchase]  Desktop  North       Organic   
2                      [Browse, Add to Cart]  Desktop  South  Social Media   
3  [Browse, Add to Cart, Checkout, Purchase]   Mobile  South       Organic   
4                                   [Browse]  Desktop   East    Google Ads   

  Product_Category  Revenue  Session_Duration Max_funnel_Stage Boounce_Flag  
0             

In [18]:
funnel_metrics = []

for i, stage in enumerate(funnel_stages):
    if i == 0:
        count = len(session_summary)
    else:
        count = len(session_summary[session_summary['Max_funnel_Stage'].isin(funnel_stages[i:])])
    funnel_metrics.append({'Stage': stage,'Session':count,'Stage_Order': i})

funnel_df = pd.DataFrame(funnel_metrics)


funnel_df['Conversion_Rate'] = round((funnel_df['Session'] / funnel_df['Session'].iloc[0]) * 100,2)
funnel_df['Drop_off_Rate'] = (
    1 - funnel_df['Session'] / funnel_df['Session'].shift(1)
) * 100

funnel_df['Drop_off_Rate'] = funnel_df['Drop_off_Rate'].round(2)
funnel_df.loc[0, 'Drop_off_Rate'] = 0

print("OVERALL FUNNEL ANALYSIS")
print(funnel_df)


purchases = session_summary[session_summary['Max_funnel_Stage'] == 'Purchase']
print("\nREVENUE ANALYSIS")
print(f"Total Revenue: ${round(purchases['Revenue'].sum(), 2)}")
print(f"Average Order Value: ${round(purchases['Revenue'].mean(), 2)}")
print(f"Total Orders: {len(purchases)}")




OVERALL FUNNEL ANALYSIS
         Stage  Session  Stage_Order  Conversion_Rate  Drop_off_Rate
0       Browse    10000            0           100.00           0.00
1  Add to Cart     7023            1            70.23          29.77
2     Checkout     4997            2            49.97          28.85
3     Purchase     2999            3            29.99          39.98

REVENUE ANALYSIS
Total Revenue: $3340947.69
Average Order Value: $1114.02
Total Orders: 2999


In [ ]:
# Create Comprehensive Funnel Visuals
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Funnel Conversion Rate", "Stage to Stage Drop-off", "Revenue by Funnel Stage", "Session Duration by Stage"),
    specs=[[{"secondary_y": False}, {"secondary_y": False}],
           [{"secondary_y": False}, {"secondary_y": False}]]
)

# Funnel Conversion Rate
fig.add_trace(go.Bar(
    x=funnel_df['Stage'], y=funnel_df['Session'], 
    text=funnel_df['Session'], textposition='auto', 
    name='Sessions', marker_color='lightblue'
), row=1, col=1)

# Stage to Stage Drop-off
fig.add_trace(go.Scatter(
    x=funnel_df['Stage'],
    y=funnel_df['Drop_off_Rate'],
    mode='lines+markers+text',
    text=funnel_df['Drop_off_Rate'].round(2),
    textposition='top center',
    name='Drop-off Rate %',
    line=dict(color='salmon', width=3),
    marker=dict(size=8)
), row=1, col=2)

# Revenue by Stage
rev_by_stage = session_summary.groupby('Max_funnel_Stage')['Revenue'].sum().reindex(funnel_stages).fillna(0)
fig.add_trace(go.Bar(
    x=rev_by_stage.index, y=rev_by_stage.values, 
    text=rev_by_stage.round(2).values,
    name='Revenue', marker_color='lightgreen'
), row=2, col=1)

# Session Duration by Stage
dur_by_stage = session_summary.groupby('Max_funnel_Stage')['Session_Duration'].mean().reindex(funnel_stages).fillna(0)
fig.add_trace(go.Bar(
    x=dur_by_stage.index, y=dur_by_stage.values, 
    text=dur_by_stage.round(2).values,
    name='Duration (m)', marker_color='mediumpurple'
), row=2, col=2)

fig.update_layout(title_text="Comprehensive Funnel Analysis Dashboard", height=800, showlegend=False)
fig.show()

In [34]:
channel_metrics = []

for channel in session_summary['Channel'].unique():
    channel_data = session_summary[session_summary['Channel'] == channel]
    total_sessions = len(channel_data)

    if total_sessions == 0:
        continue

    counts=[]
    for i ,stage in enumerate(funnel_stages):
        if i == 0:
            counts.append(total_sessions)
        else:
            valid_stages = funnel_stages[i:] 
            counts.append(len(channel_data[channel_data['Max_funnel_Stage'].isin(valid_stages)]))

    purchases = channel_data[channel_data['Max_funnel_Stage'] == 'Purchase'] 
    total_revenue = purchases['Revenue'].sum()


    aov = purchases['Revenue'].mean() if len(purchases) > 0 else 0 
    avg_duration = channel_data['Session_Duration'].mean()

    conversion_rate = (counts[-1] / total_sessions) * 100

    channel_metrics.append({
        'Channel': channel,
        'Total_Sessions': total_sessions,
        'Browse': counts[0],
        'Add_to_Cart': counts[1],
        'Checkout': counts[2],
        'Purchase': counts[3],
        'Conversion_Rate (%)': round(conversion_rate,2),
        'Total_Revenue': round(total_revenue ,2),
        'AOV': round(aov, 2),
        'Avg_Duration (m)': round(avg_duration,2)
    })

    channel_df= pd.DataFrame(channel_metrics)
    print("---Channel Performance Summary---")
    print(channel_df.to_string(index=False))


    fig2 = make_subplots(
        rows =2, cols= 2,
        subplot_titles=("Conversion Rate by Channel", "Total Revenue by Channel","Average Order Value (AOV)","Avg session Duration (MINs)")
    )

    fig2.add_trace(go.Bar(
    x=channel_df['Channel'], y=channel_df['Conversion_Rate (%)'], 
    text=channel_df['Conversion_Rate (%)'], textposition='auto', 
    marker_color='cornflowerblue'
    ) , row=1, col=1)


    fig2.add_trace(go.Bar(
    x=channel_df['Channel'], y=channel_df['Total_Revenue'], 
    text=channel_df['Total_Revenue'], textposition='auto', 
    marker_color='mediumseagreen'
    ) , row=1, col=2)

    fig2.add_trace(go.Bar(
    x=channel_df['Channel'], y=channel_df['AOV'], 
    text=channel_df['AOV'], textposition='auto', 
    marker_color='goldenrod'
    ), row=2, col=1)


    fig2.add_trace(go.Bar(
    x=channel_df['Channel'], y=channel_df['Avg_Duration (m)'], 
    text=channel_df['Avg_Duration (m)'], textposition='auto', 
    marker_color='orchid'
    ), row=2, col=2)

    fig2.update_layout(title_text="Channel Performance Dashboard", height=800, showlegend=False, template="plotly_white")
    fig2.show()


fig3 = go.Figure(
    data=[
        go.Pie(
            labels=channel_df['Channel'],
            values=channel_df['Total_Sessions'],
            textinfo='label+percent',
            hole=0.3   
        )
    ]
)

fig3.update_layout(
    title_text="Session Distribution by Channel"
)

fig3.show()


---Channel Performance Summary---
Channel  Total_Sessions  Browse  Add_to_Cart  Checkout  Purchase  Conversion_Rate (%)  Total_Revenue     AOV  Avg_Duration (m)
  Email            2442    2442         1754      1274       794                32.51      888715.87 1119.29              5.47
---Channel Performance Summary---
Channel  Total_Sessions  Browse  Add_to_Cart  Checkout  Purchase  Conversion_Rate (%)  Total_Revenue     AOV  Avg_Duration (m)
  Email            2442    2442         1754      1274       794                32.51      888715.87 1119.29              5.47
Organic            2547    2547         1772      1255       754                29.60      825718.32 1095.12              5.23
---Channel Performance Summary---
     Channel  Total_Sessions  Browse  Add_to_Cart  Checkout  Purchase  Conversion_Rate (%)  Total_Revenue     AOV  Avg_Duration (m)
       Email            2442    2442         1754      1274       794                32.51      888715.87 1119.29              5.47

In [36]:
device_metrics = []

for device in session_summary['Device'].unique():
    dev_data = session_summary[session_summary['Device'] == device]
    total_sessions = len(dev_data)
    
    if total_sessions == 0: continue
        

    counts = []
    for i, stage in enumerate(funnel_stages):
        if i == 0: 
            counts.append(total_sessions)
        else: 
            counts.append(len(dev_data[dev_data['Max_funnel_Stage'].isin(funnel_stages[i:])]))
            
    purchases = dev_data[dev_data['Max_funnel_Stage'] == 'Purchase']
    total_revenue = purchases['Revenue'].sum()
    aov = purchases['Revenue'].mean() if len(purchases) > 0 else 0
    avg_duration = dev_data['Session_Duration'].mean()
    
    device_metrics.append({
        'Device': device,
        'Total_Sessions': total_sessions,
        'Conversion_Rate': round((counts[-1] / total_sessions) * 100, 2),
        'Total_Revenue': round(total_revenue, 2),
        'AOV': round(aov, 2),
        'Avg_Duration': round(avg_duration, 2)
    })

device_df = pd.DataFrame(device_metrics)
print("--- Device Performance Summary ---")
print(device_df.to_string(index=False))



category_metrics = []

for category in session_summary['Product_Category'].unique():
    cat_data = session_summary[session_summary['Product_Category'] == category]
    total_sessions = len(cat_data)
    
    if total_sessions == 0: continue
        
    counts = []
    for i, stage in enumerate(funnel_stages):
        if i == 0: 
            counts.append(total_sessions)
        else: 
            counts.append(len(cat_data[cat_data['Max_funnel_Stage'].isin(funnel_stages[i:])]))
            
    purchases = cat_data[cat_data['Max_funnel_Stage'] == 'Purchase']
    total_revenue = purchases['Revenue'].sum()
    aov = purchases['Revenue'].mean() if len(purchases) > 0 else 0
    avg_duration = cat_data['Session_Duration'].mean()
    
    category_metrics.append({
        'Category': category,
        'Total_Sessions': total_sessions,
        'Conversion_Rate': round((counts[-1] / total_sessions) * 100, 2),
        'Total_Revenue': round(total_revenue, 2),
        'AOV': round(aov, 2),
        'Avg_Duration': round(avg_duration, 2)
    })

category_df = pd.DataFrame(category_metrics)
print("\n--- Product Category Performance Summary ---")
print(category_df.to_string(index=False))



print("\nGenerating Device & Category Dashboard...")


fig3 = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Conversion Rate by Device", "Revenue Share by Device", 
                    "Conversion Rate by Category", "Revenue Share by Category"),
    specs=[[{"type": "bar"}, {"type": "domain"}], 
           [{"type": "bar"}, {"type": "domain"}]] 
)


fig3.add_trace(go.Bar(
    x=device_df['Device'], y=device_df['Conversion_Rate'], 
    text=device_df['Conversion_Rate'], textposition='auto', marker_color='teal'
), row=1, col=1)


fig3.add_trace(go.Pie(
    labels=device_df['Device'], values=device_df['Total_Revenue'], 
    hole=0.4, name="Device Revenue"
), row=1, col=2)


fig3.add_trace(go.Bar(
    x=category_df['Category'], y=category_df['Conversion_Rate'], 
    text=category_df['Conversion_Rate'], textposition='auto', marker_color='coral'
), row=2, col=1)


fig3.add_trace(go.Pie(
    labels=category_df['Category'], values=category_df['Total_Revenue'], 
    hole=0.4, name="Category Revenue"
), row=2, col=2)

fig3.update_layout(title_text="Device & Product Category Analysis", height=800, showlegend=True, template="plotly_white")
fig3.show()

--- Device Performance Summary ---
 Device  Total_Sessions  Conversion_Rate  Total_Revenue     AOV  Avg_Duration
 Laptop            3319            30.76     1158401.39 1134.58          5.25
Desktop            3404            30.49     1136597.01 1094.99          5.36
 Mobile            3277            28.68     1045949.29 1112.71          5.17

--- Product Category Performance Summary ---
   Category  Total_Sessions  Conversion_Rate  Total_Revenue     AOV  Avg_Duration
       Home            2008            29.28      646832.68 1100.06          5.16
    Fashion            2009            28.67      639533.02 1110.30          5.24
     Sports            1982            32.04      720134.89 1134.07          5.42
Electronics            1971            29.68      643888.00 1100.66          5.20
     Beauty            2030            30.30      690559.10 1122.86          5.27

Generating Device & Category Dashboard...


In [52]:
session_summary['Date'] = session_summary['Start_time'].dt.date

daily_metrics = session_summary.groupby('Date').agg(
    Total_Sessions=('Session_ID', 'count'),
    Total_Revenue=('Revenue', 'sum'),
    Purchases=('Max_funnel_Stage', lambda x: (x == 'Purchase').sum())
).reset_index()


daily_metrics['Conversion_Rate (%)'] = round((daily_metrics['Purchases'] / daily_metrics['Total_Sessions']) * 100, 2)

session_summary['Hour'] = session_summary['Start_time'].dt.hour

hourly_metrics = session_summary.groupby('Hour').agg(
    Total_Sessions=('Session_ID', 'count'),
    Purchases=('Max_funnel_Stage', lambda x: (x == 'Purchase').sum())
).reset_index()


hourly_metrics['Conversion_Rate (%)'] = round((hourly_metrics['Purchases'] / hourly_metrics['Total_Sessions']) * 100, 2)

print("--- Top 5 Days by Revenue ---")
print(daily_metrics.sort_values('Total_Revenue', ascending=False).head().to_string(index=False))


print("\nGenerating Time-Based Dashboard...")

fig4 = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Daily Sessions Over Time", "Daily Conversion Rate", 
                    "Sessions by Hour of Day", "Conversion Rate by Hour"),
    specs=[[{"type": "scatter"}, {"type": "scatter"}], 
           [{"type": "bar"}, {"type": "bar"}]] 
)


fig4.add_trace(go.Scatter(
    x=daily_metrics['Date'], y=daily_metrics['Total_Sessions'], 
    mode='lines+markers', line=dict(color='royalblue', width=2), name="Daily Sessions"
), row=1, col=1)


fig4.add_trace(go.Scatter(
    x=daily_metrics['Date'], y=daily_metrics['Conversion_Rate (%)'], 
    mode='lines+markers', line=dict(color='darkorange', width=2), name="Daily Conv. Rate"
), row=1, col=2)


fig4.add_trace(go.Bar(
    x=hourly_metrics['Hour'], y=hourly_metrics['Total_Sessions'], 
    marker_color='lightslategray', name="Hourly Sessions"
), row=2, col=1)


fig4.add_trace(go.Bar(
    x=hourly_metrics['Hour'], y=hourly_metrics['Conversion_Rate (%)'], 
    marker_color='mediumseagreen', name="Hourly Conv. Rate"
), row=2, col=2)

fig4.update_layout(title_text="Time-Based Performance Analysis", height=800, showlegend=False, template="plotly_white")


fig4.update_xaxes(title_text="Hour of Day (0-23)", row=2, col=1)
fig4.update_xaxes(title_text="Hour of Day (0-23)", row=2, col=2)

fig4.show()

--- Top 5 Days by Revenue ---
      Date  Total_Sessions  Total_Revenue  Purchases  Conversion_Rate (%)
2026-03-19             376      138771.18        119                31.65
2026-04-01             333      132622.84        115                34.53
2026-03-27             357      127280.40        114                31.93
2026-04-08             352      124702.96        102                28.98
2026-03-26             320      118639.99        108                33.75

Generating Time-Based Dashboard...


In [58]:
total_sessions = len(session_summary)
total_purchases = len(session_summary[session_summary['Max_funnel_Stage'] == 'Purchase'])
overall_conv_rate = round((total_purchases / total_sessions) * 100, 2)
total_revenue = session_summary['Revenue'].sum()
aov = total_revenue / total_purchases if total_purchases > 0 else 0


kpi_data = {
    'Metric': [
        'Total Sessions', 
        'Total Orders (Purchases)', 
        'Overall Conversion Rate (%)', 
        'Total Revenue ($)', 
        'Average Order Value ($)'
    ],
    'Value': [
        total_sessions, 
        total_purchases, 
        overall_conv_rate, 
        round(total_revenue, 2), 
        round(aov, 2)
    ]
}
kpi_df = pd.DataFrame(kpi_data)

print("--- Executive KPIs ---")
print(kpi_df.to_string(index=False))



biggest_drop_off = funnel_df.loc[funnel_df['Drop_off_Rate'].idxmax()]


best_channel = channel_df.loc[channel_df['Total_Revenue'].idxmax()]

print("\n--- Strategic Recommendations ---")
print(f"1. Fix the '{biggest_drop_off['Stage']}' Step: This stage has a massive {biggest_drop_off['Drop_off_Rate']}% drop-off rate. UX/UI teams need to investigate this immediately.")
print(f"2. Increase Budget for {best_channel['Channel']}: This channel generated the most revenue (${best_channel['Total_Revenue']}). Reallocate marketing spend here.")



print("\nExporting final report to Excel...")

try:
    with pd.ExcelWriter('Funnel_Analysis_Report.xlsx') as writer:
        kpi_df.to_excel(writer, sheet_name='Executive KPIs', index=False)
        funnel_df.to_excel(writer, sheet_name='Funnel Metrics', index=False)
        channel_df.to_excel(writer, sheet_name='Channel Analysis', index=False)
        device_df.to_excel(writer, sheet_name='Device Analysis', index=False)
        category_df.to_excel(writer, sheet_name='Category Analysis', index=False)
        daily_metrics.to_excel(writer, sheet_name='Daily Trends', index=False)
    
    print("Success! 🚀 Report exported to 'Funnel_Analysis_Report.xlsx'")
except ModuleNotFoundError:
    print("ERROR: You need the 'openpyxl' library to export to Excel. Run 'pip install openpyxl' in your terminal and try again.")

--- Executive KPIs ---
                     Metric      Value
             Total Sessions   10000.00
   Total Orders (Purchases)    2999.00
Overall Conversion Rate (%)      29.99
          Total Revenue ($) 3340947.69
    Average Order Value ($)    1114.02

--- Strategic Recommendations ---
1. Fix the 'Purchase' Step: This stage has a massive 39.98% drop-off rate. UX/UI teams need to investigate this immediately.
2. Increase Budget for Email: This channel generated the most revenue ($888715.87). Reallocate marketing spend here.

Exporting final report to Excel...
Success! 🚀 Report exported to 'Funnel_Analysis_Report.xlsx'
